# Intermediate 04 — Secrets and Credential Security

A Northwind support agent needs policy and ticket providers that still require credentials. You will keep credential material outside the agent data plane, derive a one-use lease from authenticated workload and resource state, and materialize the exact credential version only inside a narrow executor.

## Learning objectives and safety contract

By the end, you can separate secret values, references, leases, and authorization; bind credentials to workload, sender key, tenant, audience, operation, resource, policy, and version; reject replay and stale state; preserve dependency failures; and measure secret exposure independently from utility. All identities, resources, and credential material are synthetic. The in-memory vault, HMAC lease, and Python zeroization are teaching analogues—not production vault, OAuth, HSM, or memory-erasure implementations.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, replace
from datetime import timedelta
from pathlib import Path
import json
import sys

COURSE = Path.cwd()
if not (COURSE / 'lab.py').exists():
    COURSE = Path('curriculum/roadmap/intermediate/04-secrets-and-credential-security').resolve()
assert (COURSE / 'lab.py').exists(), 'Run from the repository root or the course directory.'
sys.path.insert(0, str(COURSE))
import lab

print('Loaded course:', COURSE.name)
print('Synthetic clock:', lab.NOW.isoformat())

## 1. Put the credential boundary outside the agent

![Credential lease and execution architecture](architecture.svg)

The model proposes a business identifier. Trusted application code supplies the authenticated workload and fixed route. The broker authorizes metadata and issues a narrow handle without reading secret bytes. The executor rechecks current state, atomically claims the handle, materializes the exact version, calls one fixed provider, zeroizes its buffer, and returns only bounded content and allowlisted evidence.

In [ ]:
scenario = lab.build_scenario()
workload, _ = scenario.workloads.authenticate('attest:support', now=lab.NOW)
metadata, _ = scenario.vault.current('secret:policy-api:north')
assert metadata is not None and scenario.vault.materialization_count == 0
{
    'workload': (workload.workload_id, workload.tenant, workload.key_thumbprint),
    'secret_metadata': (metadata.secret_id, metadata.audience, metadata.version, metadata.state.value),
    'materializations': scenario.vault.materialization_count,
}

## 2. Establish the unsafe baseline

The demo-only baseline places reusable material in model context, memory, tracing, and an exception. Masked display or a prompt instruction would not undo those crossings.

In [ ]:
unsafe = lab.unsafe_context_baseline(lab._POLICY_MATERIAL_V1)
assert lab.contains_material(unsafe, (lab._POLICY_MATERIAL_V1,))
print('Unsafe baseline surfaces:', sorted(unsafe))
print('Synthetic credential exposed:', True)

## 3. Run a useful path without agent-visible material

The application derives audience, purpose, operation, and request identity. Issuance reads metadata only; materialization occurs once, after the lease is claimed, and the temporary buffer is zeroized after the provider call.

In [ ]:
scenario = lab.build_scenario()
response = scenario.application.read_policy(
    attestation_id='attest:support', policy_id='policy:north:7',
    request_id='notebook:valid', now=lab.NOW,
)
assert response.status is lab.DecisionStatus.ALLOW
assert response.content == 'Synthetic Northwind policy summary'
assert scenario.vault.materialization_count == 1
assert scenario.policy_executor.last_handle_zeroized
assert scenario.application.ambient_fallback_count == 0
controlled_surface = {'response': response, 'audit': scenario.audit.events}
assert not lab.contains_material(controlled_surface, (lab._POLICY_MATERIAL_V1,))
[(receipt.stage, receipt.status, receipt.reason) for receipt in response.receipts]

In [ ]:
lease = scenario.broker.leases[response.lease_id]
lease_contract = {
    'workload': lease.workload_id, 'tenant': lease.tenant,
    'audience': lease.audience, 'purpose': lease.purpose,
    'operations': sorted(lease.operations), 'resources': sorted(lease.resources),
    'sender_key': lease.confirmation_thumbprint,
    'secret_version': lease.secret_version, 'expires_at': lease.expires_at.isoformat(),
    'integrity_valid': scenario.broker.verify_integrity(lease),
}
assert 'material' not in asdict(lease) and 'credential' not in asdict(lease)
lease_contract

## 4. Separate providers and credentials

Policy reads and ticket writes use different audiences, secrets, operations, resources, and executors. A credential route is not a general-purpose proxy.

In [ ]:
scenario = lab.build_scenario()
policy = scenario.application.read_policy(
    attestation_id='attest:support', policy_id='policy:north:7',
    request_id='notebook:policy', now=lab.NOW,
)
ticket = scenario.application.create_ticket(
    attestation_id='attest:support', ticket_id='ticket:north:7',
    request_id='notebook:ticket', now=lab.NOW,
)
policy_lease = scenario.broker.leases[policy.lease_id]
ticket_lease = scenario.broker.leases[ticket.lease_id]
assert policy.status is ticket.status is lab.DecisionStatus.ALLOW
assert policy_lease.audience != ticket_lease.audience
assert policy_lease.secret_id != ticket_lease.secret_id
[(policy_lease.audience, sorted(policy_lease.operations)), (ticket_lease.audience, sorted(ticket_lease.operations))]

## 5. Reject audience, operation, resource, and sender confusion

Possessing a serialized lease is insufficient. The executor authenticates the workload, checks its key binding, matches the fixed provider, and enforces the exact operation and resource before any materialization.

In [ ]:
def issued_policy_lease(request_id):
    s = lab.build_scenario()
    issued = s.broker.issue('attest:support', lab.policy_request(request_id), now=lab.NOW)
    assert issued.lease is not None
    return s, issued.lease

s1, l1 = issued_policy_lease('notebook:audience')
audience = s1.ticket_executor.execute(
    'attest:support', l1.lease_id, operation='read_policy',
    resource_id='policy:north:7', request_id='notebook:audience:execute', now=lab.NOW,
)
s2, l2 = issued_policy_lease('notebook:operation')
operation = s2.policy_executor.execute(
    'attest:support', l2.lease_id, operation='write_ticket',
    resource_id='policy:north:7', request_id='notebook:operation:execute', now=lab.NOW,
)
s3, l3 = issued_policy_lease('notebook:resource')
resource = s3.policy_executor.execute(
    'attest:support', l3.lease_id, operation='read_policy',
    resource_id='policy:north:8', request_id='notebook:resource:execute', now=lab.NOW,
)
assert [d.reason for d in (audience, operation, resource)] == [
    'credential-audience', 'credential-operation', 'credential-resource'
]
[(d.status.value, d.reason) for d in (audience, operation, resource)]

In [ ]:
scenario, lease = issued_policy_lease('notebook:sender')
wrong_workload = scenario.policy_executor.execute(
    'attest:stolen', lease.lease_id, operation='read_policy',
    resource_id='policy:north:7', request_id='notebook:sender:execute', now=lab.NOW,
)
scenario, lease = issued_policy_lease('notebook:key')
original = scenario.workloads.attestations['attest:support']
scenario.workloads.attestations['attest:support'] = replace(original, key_thumbprint='key:attacker')
wrong_key = scenario.policy_executor.execute(
    'attest:support', lease.lease_id, operation='read_policy',
    resource_id='policy:north:7', request_id='notebook:key:execute', now=lab.NOW,
)
assert wrong_workload.reason == 'credential-sender'
assert wrong_key.reason == 'credential-sender-key'
[(wrong_workload.status.value, wrong_workload.reason), (wrong_key.status.value, wrong_key.reason)]

## 6. Recheck lifecycle immediately before use

Expiry, revocation, secret rotation, secret disablement, and policy-version changes invalidate an otherwise well-formed lease. Rotation is not complete until stale versions fail and a newly issued lease succeeds.

In [ ]:
scenario = lab.build_scenario()
expired_issue = scenario.broker.issue(
    'attest:support', lab.policy_request('notebook:expired', ttl=timedelta(seconds=1)), now=lab.NOW,
)
expired = scenario.policy_executor.execute(
    'attest:support', expired_issue.lease.lease_id, operation='read_policy',
    resource_id='policy:north:7', request_id='notebook:expired:execute',
    now=lab.NOW + timedelta(seconds=1),
)
scenario, lease = issued_policy_lease('notebook:revoked')
assert scenario.broker.revoke(lease.lease_id, 'incident-response')
revoked = scenario.policy_executor.execute(
    'attest:support', lease.lease_id, operation='read_policy',
    resource_id='policy:north:7', request_id='notebook:revoked:execute', now=lab.NOW,
)
assert expired.reason == 'credential-lease-expired'
assert revoked.reason == 'credential-lease-revoked'
[(expired.status.value, expired.reason), (revoked.status.value, revoked.reason)]

In [ ]:
scenario, old_lease = issued_policy_lease('notebook:pre-rotation')
new_metadata = scenario.rotate_policy_credential(now=lab.NOW + timedelta(seconds=1))
stale = scenario.policy_executor.execute(
    'attest:support', old_lease.lease_id, operation='read_policy',
    resource_id='policy:north:7', request_id='notebook:stale:execute',
    now=lab.NOW + timedelta(seconds=1),
)
fresh = scenario.application.read_policy(
    attestation_id='attest:support', policy_id='policy:north:7',
    request_id='notebook:post-rotation', now=lab.NOW + timedelta(seconds=1),
)
assert stale.reason == 'secret-version-changed'
assert fresh.status is lab.DecisionStatus.ALLOW
assert scenario.broker.leases[fresh.lease_id].secret_version == new_metadata.version
scenario.vault.disable('secret:policy-api:north')
disabled = scenario.application.read_policy(
    attestation_id='attest:support', policy_id='policy:north:7',
    request_id='notebook:disabled', now=lab.NOW + timedelta(seconds=2),
)
assert (disabled.status, disabled.reason) == (lab.DecisionStatus.DENY, 'secret-disabled')
{'stale': stale.reason, 'fresh': fresh.status.value, 'disabled': disabled.reason}

## 7. Make one-use atomic under concurrency

A check followed by a later state update is replayable. The broker claims the lease under one lock before materialization; eight simultaneous attempts therefore produce one provider call.

In [ ]:
scenario, lease = issued_policy_lease('notebook:race')
def race(index):
    return scenario.policy_executor.execute(
        'attest:support', lease.lease_id, operation='read_policy',
        resource_id='policy:north:7', request_id=f'notebook:race:{index}', now=lab.NOW,
    )
with ThreadPoolExecutor(max_workers=8) as pool:
    decisions = list(pool.map(race, range(8)))
allowed = [item for item in decisions if item.status is lab.DecisionStatus.ALLOW]
assert len(allowed) == 1
assert scenario.policy_provider.dispatch_count == 1
{'allow': len(allowed), 'denied': len(decisions) - len(allowed), 'provider_calls': scenario.policy_provider.dispatch_count}

## 8. Contain dependency failures and hostile errors

Provider exceptions are untrusted output. The executor maps arbitrary messages to a stable redacted reason. Vault and provider outages remain explicit errors; neither path falls back to environment variables, cached values, or ambient credentials.

In [ ]:
scenario = lab.build_scenario()
scenario.policy_provider.leaking_error = True
leaking = scenario.application.read_policy(
    attestation_id='attest:support', policy_id='policy:north:7',
    request_id='notebook:leaking-provider', now=lab.NOW,
)
surface = {'response': leaking, 'audit': scenario.audit.events}
assert (leaking.status, leaking.reason) == (lab.DecisionStatus.ERROR, 'provider-error-redacted')
assert not lab.contains_material(surface, (lab._POLICY_MATERIAL_V1,))
assert scenario.policy_executor.last_handle_zeroized
leaking.status.value, leaking.reason

In [ ]:
vault_scenario = lab.build_scenario()
vault_scenario.vault.available = False
vault_outage = vault_scenario.application.read_policy(
    attestation_id='attest:support', policy_id='policy:north:7',
    request_id='notebook:vault-outage', now=lab.NOW,
)
provider_scenario = lab.build_scenario()
provider_scenario.policy_provider.available = False
provider_outage = provider_scenario.application.read_policy(
    attestation_id='attest:support', policy_id='policy:north:7',
    request_id='notebook:provider-outage', now=lab.NOW,
)
assert (vault_outage.status, vault_outage.reason) == (lab.DecisionStatus.ERROR, 'secret-vault-unavailable')
assert (provider_outage.status, provider_outage.reason) == (lab.DecisionStatus.ERROR, 'provider-unavailable')
assert vault_scenario.application.ambient_fallback_count == 0
assert provider_scenario.application.ambient_fallback_count == 0
[(vault_outage.status.value, vault_outage.reason), (provider_outage.status.value, provider_outage.reason)]

## 9. Evaluate utility, misuse, exposure, stale state, replay, and failures separately

The deterministic corpus has 18 cases: 4 valid tasks, 12 attacks (11 credential-misuse probes and 1 exposure probe), and 2 dependency failures. Every metric declares its own population; error states are not counted as successful defenses.

In [ ]:
report, observations = lab.evaluate_controls()
assert (report.cases, report.valid_cases, report.attack_cases, report.failure_cases) == (18, 4, 12, 2)
assert report.valid_task_success_rate == 1.0
assert report.credential_misuse_success_rate == 0.0
assert report.secret_exposure_rate == 0.0
assert report.replay_acceptance_rate == 0.0
assert report.stale_credential_acceptance_rate == 0.0
assert report.failure_error_preservation_rate == 1.0
assert report.trace_completeness_rate == 1.0
assert report.unsafe_baseline_exposure_rate == 1.0
report

## 10. Put the real OpenAI Agents SDK behind the boundary

The repository-pinned SDK defines a strict `function_tool` whose model-facing schema contains only `policy_id`. Workload attestation, tenant, audience, purpose, operation, secret reference, lease, and request IDs stay in trusted application context. The demo invokes the real tool adapter without a model, API key, or network call.

In [ ]:
import sdk_adapter
sdk_proof = await sdk_adapter.credential_free_demo_async()
assert sdk_proof['strict_schema']
assert sdk_proof['direct_terminal_state'] == 'allow'
assert sdk_proof['denied_terminal_state'] == 'deny'
assert sdk_proof['denied_content_absent']
assert sdk_proof['real_tool_invocation_state'] == 'allow'
assert sdk_proof['trusted_fields_absent_from_schema']
assert sdk_proof['lease_ids_absent_from_results']
assert sdk_proof['raw_material_absent']
sdk_proof

## Production challenge

Replace the teaching seams with workload identity, a managed secret service or dynamic credential issuer, an opaque durable lease ledger, sender-constrained provider authentication, and an isolated executor. Preserve the executable invariants: no raw material in the agent data plane; exact tenant/audience/purpose/operation/resource/version binding; current authorization; atomic one-use claim; explicit outage behavior; bounded telemetry; and verified rotation and revocation. Add restart, partial-rotation, provider-timeout, rate-limit, and revocation-propagation tests.

## Checkpoint

An agent framework masks a provider key in object representations and stores it in local run context instead of the prompt. Is that sufficient?

**Answer:** No. Masking reduces accidental display, and local context is not automatically sent to the model, but application code can still access it and serialized run state may persist it. Keep raw material outside the agent runtime; derive a one-use lease from authenticated state; materialize the exact version only inside a narrow executor; and release only bounded results and allowlisted evidence.